# Interactive Troubleshooting Guide

**Diagnose and fix common issues**

## Learning Objectives
- Identify common error types
- Apply systematic debugging
- Use custodian for automatic error handling
- Resolve convergence issues

## Prerequisites
- Experience with atomate2siesta workflows
- Basic understanding of DFT calculations

**Estimated time: 25 minutes**

## 1. Setup

In [ ]:
from pymatgen.core import Structure, Lattice
from atomate2.siesta.jobs.core import RelaxMaker
from jobflow import run_locally

# Note: Custodian handlers are automatically included when use_custodian=True

## 2. Common Error Types

### 2.1 SCF Convergence Issues

The most common problem in DFT calculations.

In [ ]:
# Create a maker with custodian for automatic error handling
maker_with_custodian = RelaxMaker.fixed_cell_relaxation(
    dry_run=True,
    use_custodian=True,
    custodian_max_errors=10,  # Maximum error correction attempts
)

print("Custodian enabled with automatic error handlers:")
print("  - SCFConvergenceHandler: Progressive mixing/iteration strategies")
print("  - GeometryConvergenceHandler: Increase MD.NumCGsteps if needed")
print("  - WallTimeHandler: Graceful restart on time limits")
print("  - Max correction attempts: 10")
print("\nCustodian will automatically:")
print("  - Remove DM file on SCF failure")
print("  - Adjust mixing parameters")
print("  - Increase SCF iterations")
print("  - Save restart files")

### 2.2 SCF Convergence Strategies

Manual fixes for SCF issues:

In [ ]:
# Strategy 1: Reduce mixing weight (most common fix)
conservative_maker = RelaxMaker.fixed_cell_relaxation(
    dry_run=True,
    user_params={
        "SCF.Mixer.Weight": 0.02,  # More conservative (default 0.1)
        "MaxSCFIterations": 200,   # More iterations
    }
)
print("Strategy 1: Conservative mixing (0.02, default 0.1)")

# Strategy 2: Use Broyden mixer (better for difficult cases)
broyden_maker = RelaxMaker.fixed_cell_relaxation(
    dry_run=True,
    user_params={
        "SCF.Mixer.Method": "Broyden",
        "SCF.Mixer.History": 8,
    }
)
print("Strategy 2: Broyden mixer with 8-step history")

# Strategy 3: Relaxed tolerance (when close to convergence)
relaxed_maker = RelaxMaker.fixed_cell_relaxation(
    dry_run=True,
    user_params={
        "DM.Tolerance": 1.0e-3,  # Less strict (default 1e-4)
    }
)
print("Strategy 3: Relaxed DM tolerance (1e-3)")

# Strategy 4: Temperature smearing for metals
metal_maker = RelaxMaker.fixed_cell_relaxation(
    dry_run=True,
    user_params={
        "ElectronicTemperature": "300 K",
        "OccupationFunction": "MP",  # Methfessel-Paxton
    }
)
print("Strategy 4: Electronic temperature smearing for metals")

## 3. Memory Issues

In [ ]:
# Large systems may run out of memory
# Reduce memory usage with these settings:

memory_efficient_maker = RelaxMaker.fixed_cell_relaxation(
    dry_run=True,
    user_params={
        "UseSaveData": True,        # Enable restart (saves memory)
        "SaveHS": False,            # Don't save Hamiltonian (large file)
        "WriteWaveFunctions": False, # Don't write wavefunctions
        "WriteKbands": False,       # Don't write k-resolved bands
        "WriteMullikenPop": 0,      # Skip Mulliken analysis
    }
)
print("Memory-efficient settings applied")
print("These settings can reduce memory usage by 50-70%")

## 4. Debugging Checklist

In [ ]:
debugging_checklist = """
ATOMATE2SIESTA DEBUGGING CHECKLIST
===================================

1. STRUCTURE VALIDATION
   [ ] No overlapping atoms (min distance > 0.5 Å)
   [ ] Reasonable bond lengths for your material
   [ ] Correct stoichiometry and composition
   [ ] Lattice parameters make sense
   [ ] Structure is relaxed (for surfaces/interfaces)

2. PSEUDOPOTENTIAL CHECKS
   [ ] All elements have pseudopotentials installed
   [ ] XC functional matches pseudopotentials (PBE, LDA, etc.)
   [ ] Valence electron count is correct
   [ ] Pseudopotentials are in SIESTA_PP_PATH
   [ ] Check: atomate2siesta-pseudos available

3. CONVERGENCE PARAMETERS
   [ ] K-points: sufficient for your system (metals need more)
   [ ] Mesh cutoff: converged (typically 200-400 Ry)
   [ ] Basis set: appropriate (DZP minimum, TZP for accuracy)
   [ ] SCF tolerance: not too strict (1e-4 standard)

4. SCF CONVERGENCE ISSUES
   [ ] Try reducing mixing weight (0.02-0.05)
   [ ] Increase MaxSCFIterations (200+)
   [ ] Use Broyden mixer for difficult cases
   [ ] For metals: add electronic temperature smearing
   [ ] Remove old DM files (custodian does this automatically)

5. GEOMETRY RELAXATION ISSUES
   [ ] Check MD.NumCGsteps is sufficient (default 100)
   [ ] Force tolerance reasonable (0.02 eV/Å typical)
   [ ] Structure not trapped in local minimum
   [ ] Enable custodian for automatic error recovery

6. OUTPUT FILE CHECKS
   [ ] siesta.out: check for error messages
   [ ] SCF convergence history: monotonic decrease?
   [ ] Final forces: all below tolerance?
   [ ] Stress (if needed): below tolerance?
   [ ] Job completed successfully (no crashes)?

7. ENABLE AUTOMATIC FIXES
   [ ] use_custodian=True (automatic error recovery)
   [ ] custodian_max_errors=10 (default)
   [ ] dry_run=True first (preview without running)

8. COMMON FIXES
   [ ] SCF not converging → Reduce mixing weight
   [ ] Relaxation slow → Increase MD.NumCGsteps
   [ ] Out of memory → Reduce output file sizes
   [ ] Wrong results → Check convergence parameters
"""
print(debugging_checklist)

## 5. Structure Validation

In [ ]:
def validate_structure(structure):
    """Check structure for common issues."""
    issues = []
    
    # Check for too-close atoms
    for i, site_i in enumerate(structure):
        for j, site_j in enumerate(structure):
            if i < j:
                dist = site_i.distance(site_j)
                if dist < 0.5:  # Angstrom
                    issues.append(f"Atoms {i} and {j} too close: {dist:.2f} A")
    
    # Check lattice parameters
    if structure.lattice.a < 1.0:
        issues.append("Lattice parameter a very small")
    
    # Check volume
    vol_per_atom = structure.volume / len(structure)
    if vol_per_atom < 5:
        issues.append(f"Volume per atom suspiciously small: {vol_per_atom:.1f} A^3")
    
    if issues:
        print("Issues found:")
        for issue in issues:
            print(f"  - {issue}")
    else:
        print("Structure looks OK!")
    
    return len(issues) == 0

# Test with a structure
si = Structure(
    lattice=Lattice.cubic(5.43),
    species=["Si", "Si"],
    coords=[[0, 0, 0], [0.25, 0.25, 0.25]]
)

validate_structure(si)

## 6. Exercises

### Exercise 1: Diagnose SCF Issues
What parameters would you adjust for a metallic system with SCF problems?

In [ ]:
# Your answer here


### Exercise 2: Structure Validation
Add more checks to the validate_structure function.

In [ ]:
# Your code here


## Summary

Key troubleshooting strategies:
1. **Enable custodian** (`use_custodian=True`) for automatic error handling
2. **Reduce mixing weight** (0.02-0.05) for SCF convergence issues
3. **Check structure validity** before running (no overlapping atoms)
4. **Use dry-run mode** (`dry_run=True`) to test workflows
5. **Follow systematic debugging checklist** (above)

## Common Error Solutions

| Problem | Solution |
|---------|----------|
| SCF not converging | Reduce mixing weight, increase iterations |
| Geometry not relaxing | Increase MD.NumCGsteps, enable custodian |
| Out of memory | Disable unnecessary output files |
| Wrong pseudopotentials | Check XC functional matches, verify installation |
| Slow calculations | Check k-points and mesh cutoff convergence |

## Resources
- **Custodian docs**: `docs/source/custodian.rst`
- **Troubleshooting guide**: `docs/source/troubleshooting.rst`
- **Error handling tutorial**: `../03-advanced-features/03-infrastructure/03-error-handling/`
- **CLI help**: `atomate2siesta-info workflows --help`

## Advanced Debugging

For complex issues:
```python
# Enable detailed logging
import logging
logging.basicConfig(level=logging.DEBUG)

# Check parameter evolution
# See: parameter_evolution.log after running

# Use jobflow-remote for cluster debugging
# atomate2siesta-jobflow-remote -p production job inspect <job_id>
```